# 01 · 数据、标签与 14:59:59 特征合同

主目标为 `total_bad_move_bps`，我方边际冲击 guardrail 为 `impact_me_bad_bps`。本阶段只使用 14:59:59 可知信息，并把 no-order anchor 与观测训练行分开保存。

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd


def find_repo_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'outputs' / '00_active_data.json').exists():
            return candidate
    raise FileNotFoundError('run 00_get_data.ipynb first')


ROOT = find_repo_root()
OUTPUT_DIR = ROOT / 'outputs'
PROCESSED_DIR = ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
ACTIVE = json.loads((OUTPUT_DIR / '00_active_data.json').read_text(encoding='utf-8'))
panel = pd.read_parquet(ROOT / ACTIVE['files']['panel'])
panel['date'] = pd.to_datetime(panel['date'], errors='coerce').dt.normalize()
panel['sym'] = panel['sym'].astype(str)
panel['side'] = panel['side'].astype(str).str.lower().str.strip()
panel['quote_strategy'] = panel['quote_strategy'].astype(str).str.lower().str.strip()
panel['x_adv'] = pd.to_numeric(panel['x_adv'], errors='coerce')
panel['side_sign'] = np.where(panel['side'].str.startswith('sell'), -1.0, 1.0)


In [ ]:
impact_rebuilt = np.maximum(
    panel['side_sign'] * pd.to_numeric(panel['impact_me_raw_bps'], errors='coerce').fillna(0.0), 0.0
)
total_rebuilt = np.maximum(
    panel['side_sign'] * pd.to_numeric(panel['total_move_bps'], errors='coerce').fillna(0.0), 0.0
)
provided_impact = pd.to_numeric(panel['impact_me_bad_bps'], errors='coerce')
provided_total = pd.to_numeric(panel['total_bad_move_bps'], errors='coerce')
impact_gap = np.abs(provided_impact - impact_rebuilt)
total_gap = np.abs(provided_total - total_rebuilt)
if impact_gap.dropna().max() > 1e-8 or total_gap.dropna().max() > 1e-8:
    raise ValueError('public label reconstruction does not match supplied labels')
panel['impact_me_bad_bps'] = impact_rebuilt
panel['total_bad_move_bps'] = total_rebuilt

ratio_bins = [-np.inf, 0.001, 0.0025, 0.005, 0.01, 0.02, 0.05, np.inf]
ratio_labels = ['<=0.1%', '0.1-0.25%', '0.25-0.5%', '0.5-1%', '1-2%', '2-5%', '>5%']
panel['ratio_bucket'] = pd.cut(panel['x_adv'], bins=ratio_bins, labels=ratio_labels).astype(str)

feature_candidates = [
    'x_adv', 'p_limit', 'vcp_145959',
    'log_price', 'ret_vcp_3m_to_1s_bps', 'ret_vcp_60s_to_1s_bps', 'ret_vcp_30s_to_1s_bps',
    'vcp_path_vol_bps', 'vcp_path_abs_sum_bps', 'vcp_last_slope_bps', 'vcp_reversal_flag',
    'vcp_max_drawup_bps', 'vcp_max_drawdown_bps', 'book_total_notional_10k',
    'book_notional_imbalance_1s', 'book_total_vol_1s', 'book_imbalance_1s',
    'book_abs_imbalance_1s', 'book_notional_to_adv_1s', 'log_adv_5d_10k',
    'log_adv_20d_10k', 'log_adv_60d_10k', 'hist_zero_rate_20d',
    'hist_abs_mean_20d', 'hist_abs_p95_20d', 'hist_signed_mean_20d', 'tick_bps', 'side_sign',
]
feature_columns = [column for column in feature_candidates if column in panel.columns]
LEAKAGE_COLUMNS = {
    'market_move_bps', 'abs_market_move_bps', 'p_close_hist', 'p_close_cf',
    'total_move_bps', 'total_bad_move_bps', 'impact_me_raw_bps', 'impact_me_bad_bps',
    'fill_ratio', 'actual_close_vcp', 'stress_close_vcp', 'carry_impact_bps',
}
leakage_features = sorted(set(feature_columns) & LEAKAGE_COLUMNS)
if leakage_features:
    raise ValueError(f'future/outcome fields entered the feature contract: {leakage_features}')

key_columns = ['date', 'sym', 'side', 'quote_strategy']
target_columns = ['total_bad_move_bps', 'impact_me_bad_bps', 'total_move_bps', 'impact_me_raw_bps', 'market_move_bps']
model_columns = list(dict.fromkeys([*key_columns, 'ratio_bucket', *target_columns, *feature_columns]))
model_panel = panel[model_columns].copy()
model_panel['row_role'] = 'observed_candidate_training_row'
model_panel.to_parquet(PROCESSED_DIR / 'model_panel.parquet', index=False, compression='zstd')


In [ ]:
anchors = panel.sort_values([*key_columns, 'x_adv']).drop_duplicates(key_columns, keep='first').copy()
anchors['x_adv'] = 0.0
anchors['ratio_bucket'] = '<=0.1%'
anchors['anchor_total_bad_bps'] = np.nan
anchors['no_order_anchor_source'] = 'model_prediction_at_x0'
anchors['anchor_impact_bad_bps'] = 0.0
anchors['row_role'] = 'model_no_order_anchor_not_observed_outcome'
anchor_columns = list(dict.fromkeys([*key_columns, 'ratio_bucket', 'anchor_total_bad_bps', 'anchor_impact_bad_bps', 'no_order_anchor_source', *feature_columns, 'row_role']))
anchors[anchor_columns].to_parquet(PROCESSED_DIR / 'capacity_anchor_panel.parquet', index=False, compression='zstd')

label_audit = pd.DataFrame([{
    'run_mode': ACTIVE['run_mode'],
    'evaluation_scope': ACTIVE['evaluation_scope'],
    'rows': len(panel),
    'dates': panel['date'].nunique(),
    'anchors': len(anchors),
    'x_adv_points': panel['x_adv'].nunique(),
    'max_total_label_gap': float(total_gap.dropna().max()),
    'max_impact_label_gap': float(impact_gap.dropna().max()),
}])
label_audit.to_csv(OUTPUT_DIR / '01_label_audit.csv', index=False)
feature_contract = pd.DataFrame({
    'feature': feature_columns,
    'visible_at': '14:59:59',
    'role': 'model_feature',
})
feature_contract.to_csv(OUTPUT_DIR / '01_feature_contract.csv', index=False)
stage_contract = {
    'run_mode': ACTIVE['run_mode'],
    'evaluation_scope': ACTIVE['evaluation_scope'],
    'model_panel': (PROCESSED_DIR / 'model_panel.parquet').relative_to(ROOT).as_posix(),
    'capacity_anchor_panel': (PROCESSED_DIR / 'capacity_anchor_panel.parquet').relative_to(ROOT).as_posix(),
    'feature_columns': feature_columns,
    'no_order_anchor_source': 'model_prediction_at_x0',
}
(OUTPUT_DIR / '01_stage_contract.json').write_text(json.dumps(stage_contract, ensure_ascii=False, indent=2), encoding='utf-8')
print(label_audit.to_string(index=False))
print(f'features at 14:59:59: {len(feature_columns)}')
